In [ ]:
!pip install -q "gensim>=4.3.3"
# numpy 互換エラーが出たら: ランタイム > セッションを再起動 して再実行


In [ ]:
# ===== knock78: 単語埋め込みのファインチューニング(GPU) =====
# 77 との差分は ●1 の freeze=False だけ。これで埋め込み E も学習対象になる。
# padding_idx=0 は勾配もゼロにするので、PAD 行 E[0] は fine-tune 後もゼロのまま(●2 で確認)。
# 実行前に: ランタイム > ランタイムのタイプを変更 > GPU

import csv
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import gensim.downloader as api

PAD = "<PAD>"
LIMIT = 100000
BATCH_SIZE = 64
LR = 0.01
EPOCHS = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- 70: 埋め込み行列 ----
kv = api.load("word2vec-google-news-300")
vocab = kv.index_to_key[:LIMIT]
demb = kv.vector_size
E = np.zeros((len(vocab) + 1, demb), dtype=np.float32)
E[1:] = kv.vectors[:LIMIT]
word2id = {PAD: 0}
for i, w in enumerate(vocab):
    word2id[w] = i + 1


# ---- 71: データ読み込み ----
def load_sst2(path):
    with open(path, encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader)
        return [(s, l) for s, l in reader]


def text_to_ids(text, word2id):
    return [word2id[w] for w in text.split() if w in word2id]


def build_dataset(data, word2id):
    ds = []
    for s, l in data:
        ids = text_to_ids(s, word2id)
        if not ids:
            continue
        ds.append(
            {
                "text": s,
                "label": torch.tensor([float(l)]),
                "input_ids": torch.tensor(ids, dtype=torch.long),
            }
        )
    return ds


# ---- 75: collate ----
def collate(batch):
    batch = sorted(batch, key=lambda ex: ex["input_ids"].size(0), reverse=True)
    seqs = [ex["input_ids"] for ex in batch]
    labels = [ex["label"] for ex in batch]
    input_ids = nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    return {"input_ids": input_ids, "label": torch.stack(labels)}


# ---- 76+78: モデル(埋め込みを fine-tune) ----
class BoWClassifier(nn.Module):
    def __init__(self, E):
        super().__init__()
        # ●1 freeze=False: これで E も requires_grad=True になり学習される(ここが 78 の全て)。
        #    padding_idx=0 は残す → PAD 行の勾配はゼロ = E[0] はゼロのまま。
        self.emb = nn.Embedding.from_pretrained(
            torch.tensor(E), freeze=False, padding_idx=0
        )
        self.fc = nn.Linear(self.emb.embedding_dim, 1)

    def forward(self, input_ids):
        vecs = self.emb(input_ids)
        mask = input_ids != 0
        lengths = mask.sum(dim=1, keepdim=True)
        summed = vecs.sum(dim=1)
        feat = summed / lengths
        return torch.sigmoid(self.fc(feat))


def train_model(model, train_data):
    loader = DataLoader(
        train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate
    )
    criterion = nn.BCELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=LR)  # E も model.parameters() に入る
    for epoch in range(EPOCHS):
        model.train()
        total = 0.0
        for batch in loader:
            ids = batch["input_ids"].to(device)
            y = batch["label"].to(device)
            prob = model(ids)
            loss = criterion(prob, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * y.size(0)
        print(f"epoch {epoch}: loss = {total / len(train_data):.4f}")
    return model


def accuracy(model, data):
    loader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
    model.eval()
    correct = 0
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            y = batch["label"].to(device)
            prob = model(ids)
            pred = (prob >= 0.5).float()
            correct += (pred == y).sum().item()
    return correct / len(data)


# ---- 実行 ----
from google.colab import files  # noqa: E402

print("train.tsv と dev.tsv を選択してアップロード:")
files.upload()

train = build_dataset(load_sst2("train.tsv"), word2id)
dev = build_dataset(load_sst2("dev.tsv"), word2id)
print("train:", len(train), "dev:", len(dev))

model = BoWClassifier(E).to(device)

t0 = time.time()
train_model(model, train)
print("train time:", round(time.time() - t0, 1), "s")
print("dev accuracy:", accuracy(model, dev))

# ●2 Q2 の答え合わせ: fine-tune 後の埋め込みを取り出して確認。
emb_now = model.emb.weight.detach().cpu().numpy()
print("E[0](PAD)の絶対値和:", float(np.abs(emb_now[0]).sum()), " ← 0.0 ならゼロ維持(予測『非ゼロ』の答え合わせ)")
print("E 全体が動いた量:", float(np.abs(emb_now - E).sum()), " ← >0 なら fine-tune された")
# 頻出実単語で確認。'</s>'(id=1)等 訓練に出ない語は勾配ゼロで動かないので probe に不適。
tid = word2id.get("the")
print(f"'the'(id={tid})が動いた量:", float(np.abs(emb_now[tid] - E[tid]).sum()), " ← 頻出語なので >0 のはず")
